# Lesson 15b: Meta-Learning Practical

**Implementations:**
- MAML for RL
- RL² with recurrent policies
- Few-shot adaptation
- Transfer learning

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. MAML Implementation

In [ ]:
class MAMLPolicy(nn.Module):
    """Policy network for MAML."""
    
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim)
        )
    
    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

class MAML:
    """MAML for RL tasks."""
    
    def __init__(self, state_dim, action_dim, 
                 inner_lr=0.01, outer_lr=0.001, inner_steps=1):
        self.policy = MAMLPolicy(state_dim, action_dim).to(device)
        self.meta_optimizer = torch.optim.Adam(self.policy.parameters(), lr=outer_lr)
        self.inner_lr = inner_lr
        self.inner_steps = inner_steps
    
    def inner_loop(self, task_data):
        """Adapt to single task."""
        # Clone policy for task-specific adaptation
        adapted_policy = copy.deepcopy(self.policy)
        optimizer = torch.optim.SGD(adapted_policy.parameters(), lr=self.inner_lr)
        
        # Inner loop: adapt on task
        for _ in range(self.inner_steps):
            states, actions, advantages = task_data
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            advantages = torch.FloatTensor(advantages).to(device)
            
            # Policy gradient
            action_probs = adapted_policy(states)
            log_probs = torch.log(action_probs.gather(1, actions.unsqueeze(1)).squeeze())
            loss = -(log_probs * advantages).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        return adapted_policy
    
    def meta_update(self, task_batch):
        """Meta-update on batch of tasks."""
        meta_loss = 0
        
        for task_train, task_test in task_batch:
            # Adapt on training data
            adapted_policy = self.inner_loop(task_train)
            
            # Evaluate on test data
            states, actions, advantages = task_test
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            advantages = torch.FloatTensor(advantages).to(device)
            
            action_probs = adapted_policy(states)
            log_probs = torch.log(action_probs.gather(1, actions.unsqueeze(1)).squeeze())
            loss = -(log_probs * advantages).mean()
            meta_loss += loss
        
        # Meta-gradient step
        meta_loss = meta_loss / len(task_batch)
        self.meta_optimizer.zero_grad()
        meta_loss.backward()
        self.meta_optimizer.step()
        
        return meta_loss.item()

print("MAML implemented")

## 2. RL² (Recurrent Meta-RL)

In [ ]:
class RL2Policy(nn.Module):
    """Recurrent policy for meta-RL."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        # Input: state + prev_action + prev_reward
        self.lstm = nn.LSTM(state_dim + action_dim + 1, hidden_dim, batch_first=True)
        self.actor = nn.Linear(hidden_dim, action_dim)
        self.hidden_dim = hidden_dim
    
    def forward(self, state, prev_action, prev_reward, hidden=None):
        # Concatenate inputs
        x = torch.cat([state, prev_action, prev_reward.unsqueeze(-1)], dim=-1)
        
        # LSTM forward
        if hidden is None:
            hidden = self.init_hidden(state.size(0))
        
        lstm_out, hidden = self.lstm(x.unsqueeze(1), hidden)
        action_logits = self.actor(lstm_out.squeeze(1))
        
        return F.softmax(action_logits, dim=-1), hidden
    
    def init_hidden(self, batch_size):
        return (torch.zeros(1, batch_size, self.hidden_dim).to(device),
                torch.zeros(1, batch_size, self.hidden_dim).to(device))

print("RL² policy implemented")

## 3. Transfer Learning Example

In [ ]:
class TransferAgent:
    """Agent with transfer learning capabilities."""
    
    def __init__(self, state_dim, action_dim):
        # Shared feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        ).to(device)
        
        # Task-specific heads
        self.task_heads = nn.ModuleDict()
    
    def add_task(self, task_name, action_dim):
        """Add new task head."""
        self.task_heads[task_name] = nn.Linear(64, action_dim).to(device)
    
    def forward(self, state, task_name):
        features = self.feature_extractor(state)
        return F.softmax(self.task_heads[task_name](features), dim=-1)
    
    def freeze_features(self):
        """Freeze feature extractor for fine-tuning."""
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

print("Transfer agent implemented")

## Summary

**Implemented:**
1. MAML for few-shot RL adaptation
2. RL² with recurrent context encoding
3. Transfer learning with shared features

**Benefits:**
- Fast adaptation to new tasks
- Knowledge reuse across tasks
- Sample efficiency

**Next:** Lesson 16 - RLHF (Reinforcement Learning from Human Feedback)